In [26]:

import os
from langchain_openai import ChatOpenAI

# 1. Set your URL (Ensure it ends in /v1)
# Replace this string with your actual Internal URL
OPENSHIFT_MODEL_URL = "https://redhataillama-31-8b-instruct-predictor.sarhachat-mvp.svc.cluster.local:8443/v1"


# 3. Securely set them as Environment Variables so LangChain can find them automatically
os.environ["OPENAI_API_BASE"] = OPENSHIFT_MODEL_URL

# 4. Initialize the LLM connection
llm = ChatOpenAI(
    model="redhataillama-31-8b-instruct", # Make sure this matches your deployed model name exactly
    temperature=0.2, # Keep it low so the clinical bot doesn't hallucinate
    max_tokens=250
)

print(f"🔗 Successfully configured LLM connection to: {OPENSHIFT_MODEL_URL}")

🔗 Successfully configured LLM connection to: https://redhataillama-31-8b-instruct-predictor.sarhachat-mvp.svc.cluster.local:8443/v1


In [27]:
from typing import TypedDict, List, Annotated
import operator

# --- THE STATE (The Clinical Stage Tracker) ---
class TriageState(TypedDict):
    # Chat History
    messages: Annotated[List[dict], operator.add]
    
    # Stage Tracker (monitors the conversation state)
    current_stage: int # 1, 2, 3, 4, or 5
    
    # Stage 1: Initial Information Gathering
    intent: str
    gender: str
    experience: str # e.g., "previous pill user", "first time"
    
    # Stage 2: Preference Screening
    frequency: str # e.g., "daily", "monthly", "set and forget"
    side_effects: List[str] # e.g., ["no weight gain", "stop periods"]
    
    # Stage 3: Health Screening
    medical_history: List[str] # e.g., ["migraines", "high blood pressure"]
    
    # Stage 4 & 5: RAG Context & Final Outputs
    cdc_guideline_context: str
    recommendation: str
    profile_verified: bool # To handle their "profile verification" step

In [28]:
from langchain_core.messages import SystemMessage

# --- THE AI-POWERED NODES ---

def stage_1_initial_info(state: TriageState):
    print("⚙️ [SYSTEM] LLM generating Stage 1 response...")
    
    # 1. Grab the chat history from the clipboard
    messages = state.get("messages", [])
    
    # 2. Give the LLM its "Persona" and strict Stage 1 instructions
    sys_msg = SystemMessage(content=(
        "You are SARHAchat, an empathetic clinical birth control assistant. "
        "You are currently in STAGE 1. Your goal is to gather initial information. "
        "Ask the user for their gender identity and if they have used birth control before. "
        "Keep your response under 3 sentences. Be warm and welcoming. Do NOT give medical advice yet."
    ))
    
    # 3. Call the model (System Prompt + Chat History)
    response = llm.invoke([sys_msg] + messages)
    
    # 4. Save the AI's response to the clipboard. 
    # Because we used 'operator.add' in our State definition, this appends to the chat history!
    return {"messages": [response], "current_stage": 1}


def stage_2_preferences(state: TriageState):
    print("⚙️ [SYSTEM] LLM generating Stage 2 response...")
    
    messages = state.get("messages", [])
    
    sys_msg = SystemMessage(content=(
        "You are SARHAchat. You are currently in STAGE 2: Preferences. "
        "Acknowledge what the user just told you. Then, ask them how often they want "
        "to think about their birth control (e.g., daily, monthly, rarely) and if they "
        "are worried about any specific side effects. Be highly empathetic."
    ))
    
    response = llm.invoke([sys_msg] + messages)
    return {"messages": [response], "current_stage": 2}


def stage_3_health_screening(state: TriageState):
    print("⚙️ [SYSTEM] LLM generating Stage 3 response...")
    
    messages = state.get("messages", [])
    
    sys_msg = SystemMessage(content=(
        "You are SARHAchat. You are currently in STAGE 3: Health Screening. "
        "This is a critical safety step. Ask the user if they have any chronic medical "
        "conditions (like migraines, high blood pressure, or Crohn's disease) that we should "
        "keep in mind before recommending a method."
    ))
    
    response = llm.invoke([sys_msg] + messages)
    return {"messages": [response], "current_stage": 3}

# We will leave Stage 4 and 5 as stubs for a moment while we test the first three
def stage_4_recommendation(state: TriageState):
    return {"current_stage": 4}

def stage_5_profile_verification(state: TriageState):
    return {"current_stage": 5}

In [29]:
from langgraph.graph import StateGraph, START, END

# 1. Initialize the Graph using their specific TriageState
workflow = StateGraph(TriageState)

# 2. Add the 5 functional "rooms" (Nodes) we built in Cell 3
workflow.add_node("stage_1", stage_1_initial_info)
workflow.add_node("stage_2", stage_2_preferences)
workflow.add_node("stage_3", stage_3_health_screening)
workflow.add_node("stage_4", stage_4_recommendation)
workflow.add_node("stage_5", stage_5_profile_verification)

# 3. Define the Routing Logic (The "Stage Tracker")
def stage_router(state: TriageState):
    # Check the clipboard. If it's empty, default to Stage 1.
    current_stage = state.get("current_stage", 1)
    
    if current_stage == 1:
        return "stage_1"
    elif current_stage == 2:
        return "stage_2"
    elif current_stage == 3:
        return "stage_3"
    elif current_stage == 4:
        return "stage_4"
    elif current_stage == 5:
        return "stage_5"
    else:
        return END

# 4. Wire the graph together
# Every time the graph starts, it asks the stage_router where to go
workflow.add_conditional_edges(START, stage_router)

# For this initial test, we will just have each node finish its turn and pause (END)
# Later, we will add logic that loops back if the user didn't answer the question
workflow.add_edge("stage_1", END)
workflow.add_edge("stage_2", END)
workflow.add_edge("stage_3", END)
workflow.add_edge("stage_4", END)
workflow.add_edge("stage_5", END)

# 5. Compile the app!
app = workflow.compile()
print("✅ LangGraph Compiled Successfully!")

✅ LangGraph Compiled Successfully!


In [30]:
from langchain_core.messages import HumanMessage

# 1. We start a brand new patient with a blank clipboard
test_patient_state = {
    "current_stage": 1, # Start at the beginning
    "messages": [],     # No chat history yet
    "intent": "consultation",
    "gender": "",
    "experience": "",
    "frequency": "",
    "side_effects": [],
    "medical_history": [],
    "cdc_guideline_context": "",
    "recommendation": "",
    "profile_verified": False
}

print("🚀 Starting Conversation with OpenShift vLLM...\n")

# 2. We run the graph. Since there are no HumanMessages yet, 
# the LLM should instantly output the Stage 1 greeting.
result = app.invoke(test_patient_state)

# 3. Print the AI's response!
print("\n🤖 SARHAchat:")
print(result["messages"][-1].content)

🚀 Starting Conversation with OpenShift vLLM...

⚙️ [SYSTEM] LLM generating Stage 1 response...

🤖 SARHAchat:
Hello and welcome! I'm Sarha, your clinical birth control assistant. To better understand your needs, could you please share with me your gender identity and whether you've used birth control before?


In [4]:
# --- THE NODES (Stage-Specific Prompts) ---

def stage_1_initial_info(state: TriageState):
    print("[STAGE 1] LLM: Welcome. To get started, what is your gender and have you used birth control before?")
    # The LLM will eventually extract 'intent', 'gender', and 'experience' here.
    return {"current_stage": 1}

def stage_2_preferences(state: TriageState):
    print("[STAGE 2] LLM: How often do you want to think about your birth control, and are there any side effects you want to avoid?")
    # The LLM extracts 'frequency' and 'side_effects' here.
    return {"current_stage": 2}

def stage_3_health_screening(state: TriageState):
    print("[STAGE 3] LLM: To keep you safe, do you have any medical history I should know about, like migraines or high blood pressure?")
    # The LLM extracts 'medical_history' here.
    return {"current_stage": 3}

def stage_4_recommendation(state: TriageState):
    print(f"[STAGE 4] SYSTEM: Triggering RAG lookup against CDC MEC using history: {state.get('medical_history')}")
    print("[STAGE 4] LLM: Based on your preferences and health history, here is an appropriate contraception recommendation...")
    # The LLM generates the 'recommendation' here based on RAG.
    return {"current_stage": 4}

def stage_5_profile_verification(state: TriageState):
    print("[STAGE 5] LLM: Here is a summary of your profile. Does everything look correct before I generate the PDF?")
    print("[STAGE 5] SYSTEM: Generating downloadable summary.")
    # Sets profile_verified to True and ends the flow.
    return {"current_stage": 5}

In [7]:
# Create a dummy "Patient Clipboard" to test the system
test_patient_state = {
    "current_stage": 4, # We are pretending the user is already on Stage 3
    "messages": [],
    "intent": "consultation",
    "gender": "female",
    "experience": "first time",
    "frequency": "daily",
    "side_effects": ["weight gain"],
    "medical_history": [],
    "cdc_guideline_context": "",
    "recommendation": "",
    "profile_verified": False
}

print("Starting the SARHAchat LangGraph Test...\n")

# Invoke the app with our test state
result = app.invoke(test_patient_state)

print("\nTest Complete!")
print(f"The graph successfully ended with the patient at Stage: {result['current_stage']}")

Starting the SARHAchat LangGraph Test...

[STAGE 4] SYSTEM: Triggering RAG lookup against CDC MEC using history: []
[STAGE 4] LLM: Based on your preferences and health history, here is an appropriate contraception recommendation...

Test Complete!
The graph successfully ended with the patient at Stage: 4


In [51]:
import os
from langchain_openai import ChatOpenAI

# 1. Set your OpenShift internal vLLM URL
OPENSHIFT_MODEL_URL = "https://redhataillama-31-8b-instruct-predictor.sarhachat-mvp.svc.cluster.local:8443/v1"

# 2. Securely set the Environment Variable
os.environ["OPENAI_API_BASE"] = OPENSHIFT_MODEL_URL

# 3. Initialize the LLM connection
llm = ChatOpenAI(
    model="redhataillama-31-8b-instruct", 
    temperature=0.2, # Low temperature for reliable data extraction and clinical safety
    max_tokens=1500
)

print(f"✅ Successfully configured LLM connection to: {OPENSHIFT_MODEL_URL}")

✅ Successfully configured LLM connection to: https://redhataillama-31-8b-instruct-predictor.sarhachat-mvp.svc.cluster.local:8443/v1


In [52]:
from typing import TypedDict, List, Annotated
import operator

class TriageState(TypedDict):
    # Chat History
    messages: Annotated[List[dict], operator.add]
    
    # Stage Tracker
    current_stage: int
    
    # Stage 1: Initial Information Gathering
    gender: str
    experience: str
    
    # Stage 2: Preference Screening
    frequency: str
    side_effects: List[str]
    
    # Stage 3: Health Screening
    medical_history: List[str]
    
    # Stage 4 & 5: RAG Context & Final Outputs
    cdc_guideline_context: str
    recommendation: str
    profile_verified: bool

In [53]:
from pydantic import BaseModel, Field

class Stage1Data(BaseModel):
    gender: str = Field(default="", description="Strictly 1-2 words for gender (e.g., 'female', 'male'). Empty if unknown.")
    experience: str = Field(default="", description="Strictly 1-5 words summarizing past birth control use (e.g., 'used pill', 'none'). Empty if unknown.")

class Stage2Data(BaseModel):
    frequency: str = Field(default="", description="Strictly 1-3 words about how often they want to take it (e.g., 'daily', 'monthly', 'rarely'). Empty if they haven't explicitly stated a preference for the future.")
    side_effects: List[str] = Field(default_factory=list, description="List of side effects to avoid.")
    
class Stage3Data(BaseModel):
    medical_history: List[str] = Field(default_factory=list, description="List of chronic medical conditions (e.g., migraines, hypertension).")

In [54]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

def stage_1_initial_info(state: TriageState):
    print("\n⚙️ [SYSTEM] Running Stage 1 Logic...")
    messages = state.get("messages", [])
    
    # 1. EXTRACT DATA: Look at the chat history and extract facts
    if messages and isinstance(messages[-1], HumanMessage):
        extractor = llm.with_structured_output(Stage1Data)
        try:
            extracted = extractor.invoke(messages)
            # Update state with newly found data (if not already filled)
            if extracted.gender: state["gender"] = extracted.gender
            if extracted.experience: state["experience"] = extracted.experience
        except Exception as e:
            print(f"Extraction skipped/failed: {e}")

    # 2. EVALUATE & CHAT: Decide what to say based on what we know
    if not state.get("gender") or not state.get("experience"):
        instruction = (
            "You are SARHAchat, an empathetic clinical birth control assistant in Stage 1. "
            "You need to know the user's gender identity and prior birth control experience. "
            f"Currently known -> Gender: '{state.get('gender')}', Experience: '{state.get('experience')}'. "
            "Ask warmly for whatever is missing. Keep it under 3 sentences."
        )
    else:
        # STRICT BOUNDARY: Prevent the LLM from asking the next stage's questions
        instruction = (
            "You are SARHAchat. You have successfully gathered the user's gender and experience. "
            "Say EXACTLY THIS and nothing more: 'Thank you for sharing that. Now, let's talk about your lifestyle preferences.' "
            "DO NOT ask any questions."
        )
        state["current_stage"] = 2

    # 3. GENERATE RESPONSE
    sys_msg = SystemMessage(content=instruction)
    response = llm.invoke([sys_msg] + messages)
    
    # 4. RETURN UPDATED STATE
    return {
        "messages": [response], 
        "current_stage": state.get("current_stage", 1),
        "gender": state.get("gender", ""),
        "experience": state.get("experience", "")
    }

def stage_2_preferences(state: TriageState):
    print("\n⚙️ [SYSTEM] Running Stage 2 Logic...")
    messages = state.get("messages", [])
    
    if messages and isinstance(messages[-1], HumanMessage):
        extractor = llm.with_structured_output(Stage2Data)
        try:
            extracted = extractor.invoke(messages)
            if extracted.frequency: state["frequency"] = extracted.frequency
            if extracted.side_effects: state["side_effects"].extend(extracted.side_effects)
        except Exception:
            pass

    if not state.get("frequency"):
        instruction = (
            "You are in Stage 2. Ask the user how often they want to think about their birth control "
            "(e.g., daily, monthly, set-and-forget). Ask if they have side effects they want to avoid. Be empathetic."
        )
    else:
        instruction = "Thank the user for sharing their preferences. Tell them we must now do a quick health screening for safety."
        state["current_stage"] = 3

    sys_msg = SystemMessage(content=instruction)
    response = llm.invoke([sys_msg] + messages)
    
    return {
        "messages": [response], 
        "current_stage": state.get("current_stage", 2),
        "frequency": state.get("frequency", ""),
        "side_effects": state.get("side_effects", [])
    }

def stage_3_health_screening(state: TriageState):
    print("\n⚙️ [SYSTEM] Running Stage 3 Logic...")
    messages = state.get("messages", [])
    
    if messages and isinstance(messages[-1], HumanMessage):
        extractor = llm.with_structured_output(Stage3Data)
        try:
            extracted = extractor.invoke(messages)
            if extracted.medical_history: state["medical_history"].extend(extracted.medical_history)
        except Exception:
            pass

    # Medical Bypass Prompt
    base_persona = (
        "You are SARHAchat, a clinical triage assistant. You are AUTHORIZED to collect medical history. "
        "DO NOT refuse to answer. DO NOT say 'I cannot provide medical advice.' You are simply collecting "
        "data for a doctor to review later."
    )

    if not state.get("medical_history"):
        instruction = base_persona + (
            "We are in Stage 3. Ask the user if they have any chronic medical conditions "
            "(like migraines, high blood pressure, cancer, or diabetes) for our safety screening."
        )
    else:
        instruction = base_persona + "Thank the user for their health information. State that you are now analyzing the CDC guidelines."
        state["current_stage"] = 4

    sys_msg = SystemMessage(content=instruction)
    response = llm.invoke([sys_msg] + messages)
    
    return {
        "messages": [response], 
        "current_stage": state.get("current_stage", 3),
        "medical_history": state.get("medical_history", [])
    }

In [55]:
def stage_4_recommendation(state: TriageState):
    print("\n⚙️ [SYSTEM] Reached Stage 4: Recommendation Stub")
    # Later: We will use state["medical_history"] and state["frequency"] to query the CDC VectorDB here!
    return {"current_stage": 4}

def stage_5_profile_verification(state: TriageState):
    print("\n⚙️ [SYSTEM] Reached Stage 5: Profile Verification Stub")
    return {"current_stage": 5}

In [56]:
from langgraph.graph import END

def dynamic_router(state: TriageState):
    """Routes based on the completeness of the clinical data."""
    
    # 1. If we are missing initial info, route to Stage 1
    if not state.get("gender") or not state.get("experience"):
        return "stage_1"
        
    # 2. If we have initial info but missing preferences, route to Stage 2
    if not state.get("frequency"):
        return "stage_2"
        
    # 3. If we have preferences but no medical history logged, route to Stage 3
    # (Note: In a production app, you'd use a flag like 'health_checked: bool' to allow patients with zero medical conditions to pass)
    if not state.get("medical_history"): 
        return "stage_3"
        
    # 4. If all data is collected, generate recommendation
    if not state.get("recommendation"):
        return "stage_4"
        
    # 5. Finally, verify
    if not state.get("profile_verified"):
        return "stage_5"
        
    return END

In [57]:
from langgraph.graph import StateGraph, START, END

# 1. Initialize the Graph
workflow = StateGraph(TriageState)

# 2. Add the nodes
workflow.add_node("stage_1", stage_1_initial_info)
workflow.add_node("stage_2", stage_2_preferences)
workflow.add_node("stage_3", stage_3_health_screening)
workflow.add_node("stage_4", stage_4_recommendation)
workflow.add_node("stage_5", stage_5_profile_verification)

# 3. THE FIX: The Router is ONLY used at the START of a new user turn
# When you call app.invoke(), this decides where the new message should go
workflow.add_conditional_edges(START, dynamic_router)

# 4. THE FIX: Every node MUST go to END after it generates an AI response
# This breaks the infinite loop and returns the AI's question to the terminal
workflow.add_edge("stage_1", END)
workflow.add_edge("stage_2", END)
workflow.add_edge("stage_3", END)
workflow.add_edge("stage_4", END)
workflow.add_edge("stage_5", END)

# 5. Compile!
app = workflow.compile()
print("✅ LangGraph Compiled Successfully! Infinite loop squashed.")

✅ LangGraph Compiled Successfully! Infinite loop squashed.


In [58]:
# 1. Create a blank patient state
patient_state = {
    "current_stage": 1,
    "messages": [],
    "gender": "",
    "experience": "",
    "frequency": "",
    "side_effects": [],
    "medical_history": [],
    "cdc_guideline_context": "",
    "recommendation": "",
    "profile_verified": False
}

print("🚀 Starting SARHAchat Consultation Mode (Type 'quit' to exit)\n")

# Run the first invocation to get the bot's greeting
result = app.invoke(patient_state)
patient_state = result # Update our local state tracker
print(f"🤖 SARHAchat: {patient_state['messages'][-1].content}")

# Interactive loop
while True:
    user_input = input("\n🧑 You: ")
    if user_input.lower() in ["quit", "exit"]:
        print("Session ended.")
        break
        
    # Append the human message
    patient_state["messages"].append(HumanMessage(content=user_input))
    
    # Run the graph again! The router will look at the new message and data
    result = app.invoke(patient_state)
    patient_state = result
    
    print(f"\n🤖 SARHAchat: {patient_state['messages'][-1].content}")
    print(f"📊 [STATE TRACKER] Stage: {patient_state.get('current_stage')} | Gender: {patient_state.get('gender')} | Exp: {patient_state.get('experience')} | Freq: {patient_state.get('frequency')}")

🚀 Starting SARHAchat Consultation Mode (Type 'quit' to exit)


⚙️ [SYSTEM] Running Stage 1 Logic...
🤖 SARHAchat: I'm so glad you're here to talk about birth control. To better understand your needs, could you please share with me your gender identity? This will help me provide you with the most accurate and supportive information.



🧑 You:  I am a woman



⚙️ [SYSTEM] Running Stage 1 Logic...
Extraction skipped/failed: Error code: 502

🤖 SARHAchat: Thank you for sharing that with me. Now, I'd love to know a bit more about your birth control experience - have you used birth control methods before, and if so, which ones?
📊 [STATE TRACKER] Stage: 1 | Gender:  | Exp:  | Freq: 



🧑 You:  I am a woman, I have use the pill. I took it everyday.



⚙️ [SYSTEM] Running Stage 1 Logic...

🤖 SARHAchat: Thank you for sharing that. Now, let's talk about your lifestyle preferences.
📊 [STATE TRACKER] Stage: 2 | Gender: woman | Exp: birth control pill | Freq: 


...


🧑 You:  I have not I feel fine.



⚙️ [SYSTEM] Reached Stage 4: Recommendation Stub

🤖 SARHAchat: I have not I feel fine.
📊 [STATE TRACKER] Stage: 4 | Gender: woman | Exp: birth control pill | Freq: set and forget



🧑 You:  sub subs good for the tummy



⚙️ [SYSTEM] Reached Stage 4: Recommendation Stub

🤖 SARHAchat: sub subs good for the tummy
📊 [STATE TRACKER] Stage: 4 | Gender: woman | Exp: birth control pill | Freq: set and forget


KeyboardInterrupt: Interrupted by user